# EEG → ERP | samodzielny preprocessing na danych eksperymentalnych

## Ściąga=

| Co chcę zrobić? | Funkcja / własność | Co otrzymuję? |
|---|---|---|
| Wczytać surowe EEG | `mne.io.read_raw_bdf(path)` | `Raw`: kanały × czas |
| Obejrzeć metadane | `raw.info`, `raw.ch_names`, `raw.get_channel_types()` | nazwy, typy, Hz |
| Zobaczyć sygnał | `raw.plot()`, `raw.compute_psd().plot()` | przebiegi i widmo |
| Wydobyć zdarzenia z BioSemi | `mne.find_events(raw, ...)` | `events`: kolumny `[próbka, poprzednia_wartość, kod]` |
| Dodać triggermape | **Customowa funkcja `naloz_triggermap(raw, events, path)`** | `Raw` z `Annotations` |
| Sprawdzić warunki | `raw.annotations`, `mne.events_from_annotations(raw)` | opisy, `events`, `event_id` |
| Zmienić próbkowanie | `raw.resample(sfreq, events=events)` | nowy `Raw` i nowe indeksy zdarzeń |
| Dodać pozycje elektrod | `raw.set_montage(...)`, `raw.plot_sensors()` | geometria kanałów |
| Zmienić referencję | `raw.set_eeg_reference(ref_channels=...)` | EEG po referencji |
| Filtrować | `raw.filter()`, `raw.notch_filter()` | edytowany sygnał, z innym widmem |
| Pociąć EEG | `mne.Epochs(raw, events, event_id, tmin, tmax, ...)` | `Epochs`: próby × kanały × czas |
| Skorygować EOG | `mne.set_bipolar_reference()`, `EOGRegression` | epoki po korekcji ocznej |
| Poprawić baseline | `epochs.apply_baseline((tmin, 0))` | epoki z baselinem |
| Odrzucić złe próby | `epochs.drop_bad(reject={...})` | epoki w ktorych byly artefakty, wedle zastosowanego kryterium |
| Obliczyć ERP | `epochs['warunek'].average()` | `Evoked` |
| Porównać ERP | `mne.viz.plot_compare_evokeds()`, `.plot_topomap()` | przebiegi, topografie |

**Jednostki:** amplitudy EEG w MNE są w **woltach** (`75e-6` = 75 µV); filtracja w Hz; czasy w sekundach.

**Ważne:** Nie wnioskuj o przebiegu zadania wyłącznie z nazw markerów! Może być tak, że pod triggerem nazwanym w jakiś konrketny sposób stoi coś innego niż nam się wydaje.

## Pliki
**Wejście:** pliki `.bdf` i `triggerMap`.

**Wyjście:** obiekt `raw`, parametry rejestracji.

**DO ZROBIENIA:** wczytaj, sprawdź liczbę kanałów, sampling, czas rejestracji i kanały `EXG`/`Status`.

In [1]:
# Uzupełnij rozwiązanie
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from mne.preprocessing import EOGRegression

DATA = Path('.')
BDF = DATA / 'dane/ERP_z_zajec/TIMzajecia.bdf'
TRIGGERMAP = DATA / 'dane/ERP_z_zajec/triggerMap_TIMzajecia.txt'

### Co jest zapisane w BDF?

BioSemi `.bdf` zawiera **ciągłą rejestrację EEG**, nazwy kanałów i próbki w każdym z nich.

`Status` przechowuje **kody triggerów**, ale nie pełne opisy warunków (w naszej implementacji). Lista opisów do identyfikacji zdarzeń jest w osobnym `triggerMap_`.

Kanały `EXG1`–`EXG8` są dodatkowe — ich przeznaczenie zależy od sposobu podłączenia podczas eksperymentu. W naszym labowym pipeline

- `EXG1`–`EXG2` są używane do wertykalnego EOG,
- `EXG3`–`EXG4` są używane do horyzontalnego EOG,
- a `EXG7/EXG8` do referencji)

**DO ZROBIENIA:** ile trwa rejestracja? Jak wyglądają typy kanałów? Czy jest montaż? Czy plik ma już anotacje? Czy możesz odtworzyć szczegóły zadania na podstawie samego BDF?

In [2]:
# DO ZROBIENIA

## 1. Triggery i eventy

**Wejście:** `raw` z `Status`.

**Wyjście:** tablica `events` (każdy wiersz to jedno zdarzenia).

**DO ZROBIENIA:** policz eventy, zobacz pierwszych 10 kodów, wyświetl rozkład w czasie.

In [3]:
# DO ZROBIENIA

### Gotowa funkcja do nakładania triggerów

`triggerMap` zawiera jeden wiersz **na każde zdarzenie, w tej samej kolejności co w BDF**, np. `16:REACTION*0*training*0`. Pierwsza liczba to kod triggera; tekst po `:` opisuje zdarzenie i kontekst zadania. Ponieważ w naszych procedurach kody cyklicznie się powtarzają, **nie wolno** utożsamiać samego kodu z jedną nazwą. Funkcja sprawdza liczbę i kolejność kodów, a potem dodaje opisane anotacje. **Nie naprawia automatycznie błędów synchronizacji** — w razie rozbieżności przerywa działanie.

**Wejście:** `raw`, `events`, ścieżka do `.txt`.

**Wyjście:** kopia `Raw` z anotacjami.

In [4]:
def naloz_triggermap(raw, events, plik_txt):
    """Dopasuj kolejno kody z triggerMap do eventów i dodaj opisy jako Annotations."""
    wpisy = []
    for linia in Path(plik_txt).read_text(encoding='utf-8').splitlines():
        linia = linia.strip()
        if not linia or linia.startswith('#'):
            continue
        kod, opis = linia.split(':', 1)
        wpisy.append((int(kod.strip()), opis.strip().rstrip(';')))
    if len(events) != len(wpisy):
        raise ValueError(f'Niezgodna liczba triggerów: EEG={len(events)}, mapa={len(wpisy)}')
    kody_mapy = np.array([kod for kod, _ in wpisy], dtype=int)
    if not np.array_equal(events[:, 2], kody_mapy):
        i = int(np.where(events[:, 2] != kody_mapy)[0][0])
        raise ValueError(f'Niezgodna sekwencja na pozycji {i}: EEG={events[i,2]}, mapa={kody_mapy[i]}')
    opisy = [opis for _, opis in wpisy]
    onsets = (events[:, 0] - raw.first_samp) / raw.info['sfreq']
    nowe = mne.Annotations(onset=onsets, duration=np.zeros(len(onsets)), description=opisy,
                            orig_time=raw.annotations.orig_time)
    wynik = raw.copy()
    wynik.set_annotations(wynik.annotations + nowe)
    return wynik

### Funkcja `naloz_triggermap()` — wejście i wyjście

#### Wejście

- **`raw`** (`mne.io.Raw`) — obiekt zawierający sygnał EEG.
- **`events`** (`np.ndarray`) — tablica o wymiarach `(n_events, 3)`, zawierająca momenty wystąpienia i numeryczne kody triggerów.
- **`plik_txt`** (`str` lub `Path`) — ścieżka do pliku `triggerMap`, zawierającego kody i opisy zdarzeń.

Przykładowy wpis w `triggerMap`:

`16:REACTION*0*training*0`

#### Wyjście

- **`wynik`** (`mne.io.Raw`) — kopia obiektu `Raw` z dodanymi anotacjami opisującymi zdarzenia eksperymentalne.

#### Co robi funkcja?

1. Odczytuje kody i nazwy zdarzeń z pliku `triggerMap`.
2. Sprawdza zgodność liczby i kolejności triggerów z zapisem EEG.
3. Przelicza momenty wystąpienia zdarzeń z próbek na sekundy.
4. Tworzy `mne.Annotations` zawierające opisy zdarzeń.
5. Dodaje anotacje do kopii `Raw`, nie zmieniając oryginalnego obiektu.

#### Jak sprawdzić wynik?

- `raw_z_opisami.annotations` — wyświetlenie dodanych anotacji.
- `raw_z_opisami.plot()` — wizualizacja EEG wraz z oznaczeniami zdarzeń.

**Do zapamiętania:** `events` zawiera numeryczne kody zdarzeń i ich pozycje w czasie, natomiast `Annotations` pozwalają przypisać im czytelne nazwy. Funkcja łączy te informacje, nie zmieniając samego sygnału EEG.


**DO ZROBIENIA:** zastosują funkcję do nakładania triggerów; sprawdź, czy liczba anotacji się zgadza i co kryje się w nazwach. Odszukaj nazwy typów bloków oraz liczby prób.

In [4]:
# DO ZROBIENIA

**Dodatkowy trop (bez rozwiązania):** w pliku `beh_TIMzajecia.csv` sprawdź kolumny.

Zastanów się, które dane są przypisane do pojedynczej próby i czego nie da się wyczytać z samych impulsów EEG.

In [9]:
beh = pd.read_csv(DATA / 'dane/ERP_z_zajec/beh_TIMzajecia.csv')
print('CSV:', beh.shape, '\nKolumny:', beh.columns.tolist())
print(beh.groupby('block_type').size())
print(beh.head())

CSV: (145, 8) 
Kolumny: ['block_num', 'trial_num', 'rt', 'allowed_error', 'block_type', 'iti_time', 'feedback', 'acc']
block_type
color         45
facesimple    45
number        45
training      10
dtype: int64
   block_num  trial_num                  rt  allowed_error block_type  \
0          0          0  0.4989430904388428            100   training   
1          0          1  1.8824546337127686            110   training   
2          0          2  0.6288189888000488            120   training   
3          0          3   1.152104377746582            130   training   
4          0          4  0.6959309577941895            140   training   

   iti_time feedback  acc  
0  0.700234      neg    0  
1  0.768453      neg    0  
2  0.927477      neg    0  
3  1.034939      neg    0  
4  1.235572      neg    0  


## Próbkowanie, pozycje elektrod i referencja

**Wejście:** `raw_named` i `raw_events`. **Wyjście:** `raw_rs`, `events_rs`, potem `raw_ref`.

**DO ZROBIENIA:** zredukuj sampling **do 256 Hz razem ze zdarzeniami**, ustaw montaż i referencję zgodną z założeniami labowymi (`EXG7`, `EXG8`).

Najpierw sprawdź, o ile zmieniły się indeksy próbek i czy czas zdarzeń pozostał prawie taki sam.

In [5]:
# DO ZROBIENIA

## 3. Filtrowanie

**Wejście:** `raw_ref`.

**Wyjście:** `raw_filt`.

**DO ZROBIENIA:** ustaw filtry zgodne z pipeline badawczym: HP **0.05 Hz**, LP **25 Hz** oraz notch **50 Hz**.

Zobacz ten sam fragment przed/po i porównaj PSD.

In [6]:
# DO ZROBIENIA

In [7]:
# DO ZROBIENIA

## 4. Epoki: od ciągłego EEG do prób

**Wejście:** `raw_filt` z opisanymi anotacjami.

**Wyjście:** `epochs` (próby × kanały × punkty czasowe).

**DO ZROBIENIA:** odzyskaj zdarzenia z anotacji, wybierz zdarzenia `FEEDBACK_*`, **bez treningu**, utwórz epoki **−0.25 do 0.95 s**. Dodaj *baseline od -250ms* do pojawienia się bodźca feedback, bo najpierw skorygujemy EOG. Dlaczego nie epokujemy wokół wszystkich triggerów, np. `FIXATION` i `REACTION`?

In [8]:
# DO ZROBIENIA

## 5. Korekcja sygnału ocznego — krótko

W tym eksperymencie zapisano kanały dodatkowe, z których składamy **VEOG** i **HEOG**.

`EOGRegression` modeluje wkład tych sygnałów w EEG i usuwa go.

**DO ZROBIENIA:** uruchom korekcję i porównaj jeden fragment/czujnik przed i po. Nie trzeba na tych zajęciach ręcznie implementować regresji.

**Wejście:** `epochs`. **Wyjście:** `epochs_eog`.

In [9]:
# DO ZROBIENIA

## 6. Baseline, kontrola artefaktów i ERP

**DO ZROBIENIA:** zastosuj baseline **−0.25 do 0 s** po EOG. Obejrzyj epoki. Odrzuć epoki w których amplituda wychodzi poza ±75 µV  i zanotuj, ile prób zostało. Następnie porównaj ERP dla pozytywnej i negatywnej informacji zwrotnej.

In [10]:
# DO ZROBIENIA